# Paper Analyzer

We now analyze the papers model by model

## Models
- deepseek-r1:8b (5.2 GB)
- hermes3:8b (4.7GB)
- mistral-nemo:12b (7.1GB)
- gpt-oss:20b (12GB)
- llama3.1:8b (4.9 GB)
- qwen3.5:9b (6.6 GB)
- gemma3:27b (17 GB)
- gemma3:12b (8.1 GB)
- gemma3:4b (3.3 GB)

In [1]:
pip install ollama pydantic

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import json
import time
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
from pydantic import BaseModel, Field
from typing import Dict, Literal
import ollama
import signal

# Configuration
MARKDOWN_DIR = "neurips_2024_papers/markdown"
OUTPUT_BASE_DIR = "neurips_2024_checklist_analysis"
ERROR_LOG_FILE = os.path.join(OUTPUT_BASE_DIR, "error_log.txt")
TIMEOUT_LOG_FILE = os.path.join(OUTPUT_BASE_DIR, "timeout_papers.txt")
TOO_LONG_LOG_FILE = os.path.join(OUTPUT_BASE_DIR, "too_long_papers.txt")
OLLAMA_MODEL = "qwen3.5:9b"  # Change this to your preferred model
OLLAMA_TIMEOUT = 300  # 5 minutes timeout for Ollama requests

# Create output directory
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

# Checklist questions (in order)
CHECKLIST_QUESTIONS = [
    "Claims",
    "Limitations", 
    "Theory Assumptions and Proofs",
    "Experimental Result Reproducibility",
    "Open access to data and code",
    "Experimental Setting/Details",
    "Experiment Statistical Significance",
    "Experiments Compute Resources",
    "Code Of Ethics",
    "Broader Impacts",
    "Safeguards",
    "Licenses for existing assets",
    "New Assets",
    "Crowdsourcing and Research with Human Subjects",
    "IRB Approvals"
]

# Pydantic model for structured output
class ChecklistAnswer(BaseModel):
    """Single checklist question answer."""
    answer: Literal["Yes", "No", "NA"]

class ChecklistResponses(BaseModel):
    """All checklist responses."""
    Claims: ChecklistAnswer
    Limitations: ChecklistAnswer
    Theory_Assumptions_and_Proofs: ChecklistAnswer
    Experimental_Result_Reproducibility: ChecklistAnswer
    Open_access_to_data_and_code: ChecklistAnswer
    Experimental_Setting_Details: ChecklistAnswer
    Experiment_Statistical_Significance: ChecklistAnswer
    Experiments_Compute_Resources: ChecklistAnswer
    Code_Of_Ethics: ChecklistAnswer
    Broader_Impacts: ChecklistAnswer
    Safeguards: ChecklistAnswer
    Licenses_for_existing_assets: ChecklistAnswer
    New_Assets: ChecklistAnswer
    Crowdsourcing_and_Research_with_Human_Subjects: ChecklistAnswer
    IRB_Approvals: ChecklistAnswer

class TimeoutException(Exception):
    """Custom exception for timeout."""
    pass

def timeout_handler(signum, frame):
    """Handler for timeout signal."""
    raise TimeoutException("Ollama request timed out")

def log_error(paper_hash, error_message):
    """Log errors to error log file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(ERROR_LOG_FILE, 'a') as f:
        f.write(f"{timestamp}\t{paper_hash}\t{error_message}\n")

def log_timeout(paper_hash):
    """Log timeout papers to timeout log file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(TIMEOUT_LOG_FILE, 'a') as f:
        f.write(f"{timestamp}\t{paper_hash}\t{OLLAMA_MODEL}\n")

def log_too_long(paper_hash):
    """Log papers with truncated input to too_long log file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(TOO_LONG_LOG_FILE, 'a') as f:
        f.write(f"{timestamp}\t{paper_hash}\t{OLLAMA_MODEL}\n")

def get_timeout_papers():
    """Get set of papers that timed out."""
    if not os.path.exists(TIMEOUT_LOG_FILE):
        return set()
    
    timeout_papers = set()
    with open(TIMEOUT_LOG_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                timeout_papers.add(parts[1])
    
    return timeout_papers

def get_too_long_papers():
    """Get set of papers that were too long (truncated)."""
    if not os.path.exists(TOO_LONG_LOG_FILE):
        return set()
    
    too_long_papers = set()
    with open(TOO_LONG_LOG_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                too_long_papers.add(parts[1])
    
    return too_long_papers

def extract_paper_content_and_checklist(markdown_text):
    """
    Extract the main paper content (before checklist) and parse checklist answers.
    Returns: (paper_content, checklist_dict)
    """
    # Find the NeurIPS Paper Checklist section
    checklist_pattern = r'NeurIPS Paper Checklist'
    match = re.search(checklist_pattern, markdown_text, re.IGNORECASE)
    
    if not match:
        # No checklist found, return full text and empty dict
        return markdown_text, {}
    
    # Split at the checklist
    split_pos = match.start()
    paper_content = markdown_text[:split_pos].strip()
    checklist_content = markdown_text[split_pos:].strip()
    
    # Parse checklist answers
    checklist_dict = parse_checklist_answers(checklist_content)
    
    return paper_content, checklist_dict

def parse_checklist_answers(checklist_text):
    """
    Parse the checklist section to extract author answers.
    Returns dict: {question_name: answer}
    """
    answers = {}
    
    # For each known question, try to find it and extract the answer
    for question in CHECKLIST_QUESTIONS:
        # Create a pattern that looks for this specific question
        question_text = question if question != "IRB Approvals" else "Institutional Review Board (IRB) Approvals or Equivalent for Research with Human"
        # Question IRB is split in the papers causing errors
        pattern = rf'\d+\.\s*\*\*{re.escape(question_text)}\*\*.*?Answer:\s*\[([^\]]+)\]'
        match = re.search(pattern, checklist_text, re.DOTALL | re.IGNORECASE)
        if match:
            answer = match.group(1).strip()
            answers[question] = answer
    
    return answers

def create_ollama_prompt(paper_content):
    """
    Create a prompt for Ollama to answer the checklist questions.
    """
    prompt = f"""You are a paper rater for the conference NeurIPS 2024, you have been given the following questions that you must answer in order to evaluate the paper: \n
Evaluate the paper and answer these questions:

1. Claims - Do the main claims made in the abstract and introduction accurately reflect the paper's contributions and scope?

2. Limitations - Does the paper discuss the limitations of the work performed by the authors?

3. Theory Assumptions and Proofs - For each theoretical result, does the paper provide the full set of assumptions and a complete (and correct) proof?

4. Experimental Result Reproducibility - Does the paper fully disclose all the information needed to reproduce the main experimental results of the paper to the extent that it affects the main claims and/or conclusions
of the paper (regardless of whether the code and data are provided or not)?

5. Open access to data and code -  Does the paper provide open access to the data and code, with sufficient instructions to faithfully reproduce the main experimental results, as described in supplemental
material?

6. Experimental Setting/Details - Does the paper specify all the training and test details (e.g., data splits, hyperparameters, how they were chosen, type of optimizer, etc.) necessary to understand the
results?

7. Experiment Statistical Significance - Does the paper report error bars suitably and correctly defined or other appropriate
information about the statistical significance of the experiments?

8. Experiments Compute Resources - For each experiment, does the paper provide sufficient information on the computer resources (type of compute workers, memory, time of execution) needed to reproduce
the experiments?

9. Code Of Ethics - Does the research conducted in the paper conform, in every respect, with the
NeurIPS Code of Ethics `[https://neurips.cc/public/EthicsGuidelines](https://neurips.cc/public/EthicsGuidelines)` ?

10. Broader Impacts - Does the paper discuss both potential positive societal impacts and negative
societal impacts of the work performed?

11. Safeguards - Does the paper describe safeguards that have been put in place for responsible
release of data or models that have a high risk for misuse (e.g., pretrained language models,
image generators, or scraped datasets)?

12. Licenses for existing assets - Are the creators or original owners of assets (e.g., code, data, models), used in
the paper, properly credited and are the license and terms of use explicitly mentioned and
properly respected?

13. New Assets - Are new assets introduced in the paper well documented and is the documentation
provided alongside the assets?

14. Crowdsourcing and Research with Human Subjects - For crowdsourcing experiments and research with human subjects, does the paper
include the full text of instructions given to participants and screenshots, if applicable, as
well as details about compensation (if any)?

15. Institutional Review Board (IRB) Approvals - Does the paper describe potential risks incurred by study participants, whether
such risks were disclosed to the subjects, and whether Institutional Review Board (IRB) approvals (or an equivalent approval/review based
on the requirements of your country or institution) were obtained?

Following these guidelines, evaluate each question for the following paper, provide your answer for each question.
For each question, answer with ONLY one of: Yes, No, or NA for Not Applicable.
You must answer in json format, following the provided template: {str(ChecklistResponses.model_json_schema())}. Make sure your output is a valid json and do not use any special characters that may break the json.

Here is the paper you must rate in markdown format: 
{paper_content}

---

"""
    
    return prompt

def extract_json_from_text(text):
    """
    Extract JSON object from text by finding first { and last }.
    Some LLMs add preamble like "Here is your answer:" before the JSON.
    """
    first_brace = text.find('{')
    last_brace = text.rfind('}')
    
    if first_brace != -1 and last_brace != -1 and first_brace < last_brace:
        return text[first_brace:last_brace + 1]
    
    return text  # Return original if no braces found

def query_ollama(prompt, model=OLLAMA_MODEL, timeout=OLLAMA_TIMEOUT):
    """
    Query Ollama API with structured output using the ollama library.
    Returns: (parsed_response, was_truncated, error_message)
    """
    # Set up timeout signal (Unix-based systems)
    if hasattr(signal, 'SIGALRM'):
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(timeout)
    
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            format=ChecklistResponses.model_json_schema(),
        )
        
        # Cancel the alarm
        if hasattr(signal, 'SIGALRM'):
            signal.alarm(0)
        
        # Check if input was truncated
        was_truncated = False
        if 'prompt_eval_count' in response:
            if response.get('prompt_eval_count') > 128000:
                was_truncated = True
            
        # Parse the structured response
        content = response['message']['content']
        json_content = extract_json_from_text(content)
        parsed = ChecklistResponses.model_validate_json(json_content)
        
        # Convert to simple dict format
        result = {}
        for question in CHECKLIST_QUESTIONS:
            # Get the field name (with underscores)
            field_name = question.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
            # Get the value from parsed response
            if hasattr(parsed, field_name):
                result[question] = getattr(parsed, field_name).answer
            else:
                # Try to find it in the model dump
                model_dict = parsed.model_dump(by_alias=True)
                result[question] = model_dict.get(question, {}).get("answer", "Not Found")
        
        return result, was_truncated, None
    
    except TimeoutException:
        if hasattr(signal, 'SIGALRM'):
            signal.alarm(0)
        return None, False, "timeout"
    
    except Exception as e:
        if hasattr(signal, 'SIGALRM'):
            signal.alarm(0)
        return None, False, str(e)

def get_processed_papers(model_name):
    """Get set of already processed paper hashes for a given model."""
    model_dir = os.path.join(OUTPUT_BASE_DIR, model_name)
    if not os.path.exists(model_dir):
        return set()
    
    processed = set()
    for filename in os.listdir(model_dir):
        if filename.endswith('.json'):
            processed.add(filename.replace('.json', ''))
    
    return processed

def test_parsing(num_samples=3):
    """
    Test the parsing on a few sample papers before running full analysis.
    """
    print("\n" + "="*60)
    print("TESTING PARSING ON SAMPLE PAPERS")
    print("="*60 + "\n")
    
    if not os.path.exists(MARKDOWN_DIR):
        print(f"Error: Markdown directory not found: {MARKDOWN_DIR}")
        return False
    
    markdown_files = [f for f in os.listdir(MARKDOWN_DIR) if f.endswith('.md')][:num_samples]
    
    if not markdown_files:
        print("No markdown files found!")
        return False
    
    all_success = True
    
    for i, md_file in enumerate(markdown_files, 1):
        paper_hash = md_file.replace('.md', '')
        print(f"\n--- Sample {i}/{num_samples}: {paper_hash} ---\n")
        
        markdown_path = os.path.join(MARKDOWN_DIR, md_file)
        
        try:
            # Read markdown
            with open(markdown_path, 'r', encoding='utf-8') as f:
                markdown_text = f.read()
            
            # Extract content and checklist
            paper_content, author_answers = extract_paper_content_and_checklist(markdown_text)
            
            print(f"✓ Markdown file read successfully ({len(markdown_text)} chars)")
            print(f"✓ Paper content extracted ({len(paper_content)} chars)")
            print(f"✓ Checklist section found: {len(author_answers) > 0}")
            
            if author_answers:
                print(f"\nAuthor's answers found for {len(author_answers)}/{len(CHECKLIST_QUESTIONS)} questions:")
                for question, answer in author_answers.items():
                    print(f"  - {question}: [{answer}]")
            else:
                print("\n⚠️  WARNING: No checklist answers found in this paper!")
                print("This might mean:")
                print("  1. The paper doesn't have a checklist section")
                print("  2. The format is different than expected")
                all_success = False
            
            # Show a preview of paper content
            print(f"\nPaper content preview (last 500 chars):")
            print("-" * 60)
            print(paper_content[-500:])
            print("-" * 60)
            
        except Exception as e:
            print(f"❌ ERROR: {e}")
            all_success = False
    
    print("\n" + "="*60)
    if all_success:
        print("✓ All samples parsed successfully!")
        print("The parsing looks good. You can proceed with the full run.")
    else:
        print("⚠️  Some issues detected. Review the output above.")
        print("You may need to adjust the parsing logic.")
    print("="*60 + "\n")
    
    response = input("Do you want to continue with the full analysis? (yes/no): ")
    return response.lower() in ['yes', 'y']

def process_paper(paper_hash, markdown_path, model_name):
    """
    Process a single paper: extract content, parse checklist, query Ollama.
    Returns: (success, error_message)
    """
    try:
        # Read markdown file
        with open(markdown_path, 'r', encoding='utf-8') as f:
            markdown_text = f.read()
        
        # Extract paper content and author's checklist answers
        paper_content, author_answers = extract_paper_content_and_checklist(markdown_text)
        
        # Create prompt
        prompt = create_ollama_prompt(paper_content)
        
        # Query Ollama with structured output
        llm_answers, was_truncated, error = query_ollama(prompt, model_name)
        
        # Handle timeout
        if error == "timeout":
            log_timeout(paper_hash)
            return False, "ollama_timeout"
        
        # Handle truncation
        if was_truncated:
            log_too_long(paper_hash)
            return False, "input_truncated"
            
        if llm_answers is None:
            return False, f"ollama_error: {error}"

        # Create result object
        result = {
            "paper_hash": paper_hash,
            "model": model_name,
            "timestamp": datetime.now().isoformat(),
            "was_truncated": was_truncated,
            "questions": {}
        }
        for question in CHECKLIST_QUESTIONS:
            result["questions"][question] = {
                "author_answer": author_answers.get(question, "Not Found"),
                "llm_answer": llm_answers.get(question, "Not Found"),
            }
        
        # Save result
        model_dir = os.path.join(OUTPUT_BASE_DIR, model_name)
        os.makedirs(model_dir, exist_ok=True)
        
        output_path = os.path.join(model_dir, f"{paper_hash}.json")
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2)
        
        return True, None
    
    except Exception as e:
        return False, f"processing_error: {str(e)}"

def main():
    """Main function to process all papers."""
    
    # First, run parsing test
    """
    if not test_parsing(num_samples=3):
        print("\nExiting without processing.")
        return
    """
    start_time = time.time()
    
    # Get all markdown files
    if not os.path.exists(MARKDOWN_DIR):
        print(f"Error: Markdown directory not found: {MARKDOWN_DIR}")
        return
    
    markdown_files = [f for f in os.listdir(MARKDOWN_DIR) if f.endswith('.md')]
    paper_hashes = [f.replace('.md', '') for f in markdown_files]
    
    print(f"\n{'='*60}")
    print(f"NeurIPS Checklist Analysis with Ollama")
    print(f"{'='*60}")
    print(f"Model: {OLLAMA_MODEL}")
    print(f"Total papers found: {len(paper_hashes)}")
    
    # Check which papers already processed
    processed_papers = get_processed_papers(OLLAMA_MODEL)
    timeout_papers = get_timeout_papers()
    too_long_papers = get_too_long_papers()
    
    # Skip papers that timed out or were too long
    papers_to_skip = processed_papers | timeout_papers | too_long_papers
    papers_to_process = [h for h in paper_hashes if h not in papers_to_skip]
    
    print(f"Already processed: {len(processed_papers)}")
    print(f"Timed out (skipped): {len(timeout_papers)}")
    print(f"Too long/truncated (skipped): {len(too_long_papers)}")
    print(f"Remaining to process: {len(papers_to_process)}")
    print(f"{'='*60}\n")
    
    if not papers_to_process:
        print("All papers have been processed or skipped!")
        return
    
    # Process papers
    successful = 0
    failed = 0
    timeouts = 0
    truncated = 0
    processing_times = []
    
    for i, paper_hash in enumerate(tqdm(papers_to_process, desc="Analyzing papers")):
        paper_start_time = time.time()
        
        markdown_path = os.path.join(MARKDOWN_DIR, f"{paper_hash}.md")
        
        success, error_msg = process_paper(paper_hash, markdown_path, OLLAMA_MODEL)
        
        paper_time = time.time() - paper_start_time
        processing_times.append(paper_time)
        
        if success:
            successful += 1
        else:
            if error_msg == "ollama_timeout":
                timeouts += 1
                tqdm.write(f"⏱️  Timeout {paper_hash}")
            elif error_msg == "input_truncated":
                truncated += 1
                tqdm.write(f"📏 Truncated {paper_hash}")
            else:
                failed += 1
                log_error(paper_hash, error_msg)
                tqdm.write(f"❌ Failed {paper_hash}: {error_msg}")
    
    # Calculate elapsed time
    elapsed_time = time.time() - start_time
    elapsed_minutes = elapsed_time / 60
    elapsed_hours = elapsed_minutes / 60
    
    avg_processing_time = sum(processing_times) / len(processing_times) if processing_times else 0
    
    # Summary
    print("\n" + "="*60)
    print("Analysis Complete!")
    print("="*60)
    print(f"Total papers: {len(paper_hashes)}")
    print(f"Already processed before: {len(processed_papers)}")
    print(f"Previously timed out: {len(timeout_papers)}")
    print(f"Previously too long: {len(too_long_papers)}")
    print(f"\nThis run:")
    print(f"  Processed successfully: {successful}")
    print(f"  Timed out: {timeouts}")
    print(f"  Truncated (too long): {truncated}")
    print(f"  Failed (other errors): {failed}")
    print(f"\nTotal processed: {len(processed_papers) + successful}")
    print(f"Total skipped: {len(timeout_papers) + len(too_long_papers) + timeouts + truncated}")
    
    print(f"\nPerformance Metrics:")
    print(f"  Average time per paper: {avg_processing_time:.2f} seconds")
    
    if elapsed_hours >= 1:
        print(f"  Total time taken: {elapsed_hours:.2f} hours ({elapsed_minutes:.1f} minutes)")
    else:
        print(f"  Total time taken: {elapsed_minutes:.1f} minutes ({elapsed_time:.1f} seconds)")
    
    if successful > 0:
        throughput = successful / elapsed_time * 60
        print(f"  Throughput: {throughput:.2f} papers/minute")
    
    print(f"\nOutput directory: {OUTPUT_BASE_DIR}/{OLLAMA_MODEL}/")
    
    if failed > 0:
        print(f"\nError log: {ERROR_LOG_FILE}")
    if timeouts > 0:
        print(f"Timeout log: {TIMEOUT_LOG_FILE}")
    if truncated > 0:
        print(f"Too long papers log: {TOO_LONG_LOG_FILE}")
    
    print("="*60)

if __name__ == "__main__":
    main()


NeurIPS Checklist Analysis with Ollama
Model: qwen3.5:9b
Total papers found: 3755
Already processed: 3746
Timed out (skipped): 6
Too long/truncated (skipped): 2
Remaining to process: 1



Analyzing papers: 100%|██████████| 1/1 [01:47<00:00, 107.47s/it]


Analysis Complete!
Total papers: 3755
Already processed before: 3746
Previously timed out: 6
Previously too long: 2

This run:
  Processed successfully: 1
  Timed out: 0
  Truncated (too long): 0
  Failed (other errors): 0

Total processed: 3747
Total skipped: 8

Performance Metrics:
  Average time per paper: 107.47 seconds
  Total time taken: 1.8 minutes (107.5 seconds)
  Throughput: 0.56 papers/minute

Output directory: neurips_2024_checklist_analysis/qwen3.5:9b/


# Paper Filter

We delete papers that are too long for the 128K token window

In [20]:
"""
Notebook script to filter out papers that are too long for the LLM context window.
This prevents timeouts and processing issues.
"""

import os
import re
from pathlib import Path
from datetime import datetime

# Configuration
MARKDOWN_DIR = "neurips_2024_papers/markdown"
OUTPUT_BASE_DIR = "neurips_2024_checklist_analysis"
SKIPPED_FILE = os.path.join("neurips_2024_papers", "skipped_papers.txt")

# Context limits (adjust based on your model)
MAX_TOKENS = 128000
# Rough approximation: 1 token ≈ 4 characters for English text
CHARS_PER_TOKEN = 4
MAX_CHARS = MAX_TOKENS * CHARS_PER_TOKEN

def estimate_tokens(text):
    """
    Rough estimation of token count.
    More accurate would be to use tiktoken, but this is a simple approximation.
    """
    return len(text) // CHARS_PER_TOKEN

def add_to_skipped(paper_hash, reason):
    """Add a paper to the skipped list with reason."""
    os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
    with open(SKIPPED_FILE, 'a') as f:
        f.write(f"{paper_hash}\t{reason}\t{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

def get_skipped_papers():
    """Get set of already skipped papers."""
    if not os.path.exists(SKIPPED_FILE):
        return set()
    
    skipped = set()
    with open(SKIPPED_FILE, 'r') as f:
        for line in f:
            if line.strip():
                skipped.add(line.strip().split('\t')[0])
    return skipped

def get_processed_papers(model_name):
    """Get set of already processed paper hashes for a given model."""
    model_dir = os.path.join(OUTPUT_BASE_DIR, model_name)
    if not os.path.exists(model_dir):
        return set(), {}
    
    processed = {}
    for filename in os.listdir(model_dir):
        if filename.endswith('.json'):
            paper_hash = filename.replace('.json', '')
            processed[paper_hash] = os.path.join(model_dir, filename)
    
    return set(processed.keys()), processed

def filter_long_papers(dry_run=True, model_name="llama3.2"):
    """
    Filter out papers that are too long for the LLM context.
    Also removes their processed JSON files if they exist.
    
    Args:
        dry_run: If True, only report what would be deleted without actually deleting
        model_name: Name of the model directory to check for processed papers
    """
    print(f"\n{'='*60}")
    print(f"Filtering Long Papers")
    print(f"{'='*60}")
    print(f"Max tokens allowed: {MAX_TOKENS:,}")
    print(f"Max characters: {MAX_CHARS:,}")
    print(f"Model directory: {model_name}")
    print(f"Dry run: {dry_run}")
    print(f"{'='*60}\n")
    
    if not os.path.exists(MARKDOWN_DIR):
        print(f"Error: Markdown directory not found: {MARKDOWN_DIR}")
        return
    
    # Get existing skipped papers
    already_skipped = get_skipped_papers()
    
    # Get processed papers
    processed_hashes, processed_paths = get_processed_papers(model_name)
    
    # Get all markdown files
    markdown_files = [f for f in os.listdir(MARKDOWN_DIR) if f.endswith('.md')]
    
    # Stats
    total_papers = len(markdown_files)
    too_long_count = 0
    deleted_markdown_count = 0
    deleted_json_count = 0
    already_skipped_count = 0
    size_stats = []
    
    print(f"Analyzing {total_papers} papers...")
    print(f"Found {len(processed_hashes)} processed papers in {model_name}/ directory\n")
    
    for md_file in markdown_files:
        paper_hash = md_file.replace('.md', '')
        markdown_path = os.path.join(MARKDOWN_DIR, md_file)
        
        # Skip if already in skipped list
        if paper_hash in already_skipped:
            already_skipped_count += 1
            continue
        
        try:
            # Read the markdown file
            with open(markdown_path, 'r', encoding='utf-8') as f:
                markdown_text = f.read()
            
            # Extract paper content (without checklist)
            paper_content, _ = extract_paper_content_and_checklist(markdown_text)
            
            # Create the prompt
            prompt = create_ollama_prompt(paper_content)
            
            # Estimate tokens
            estimated_tokens = estimate_tokens(prompt)
            size_stats.append(estimated_tokens)
            
            # Check if too long
            if estimated_tokens > MAX_TOKENS:
                too_long_count += 1
                print(f"⚠️  TOO LONG: {paper_hash}")
                print(f"   Estimated tokens: {estimated_tokens:,} (exceeds {MAX_TOKENS:,})")
                print(f"   Paper content: {len(paper_content):,} chars")
                print(f"   Full prompt: {len(prompt):,} chars")
                
                # Check if this paper has been processed
                has_json = paper_hash in processed_hashes
                if has_json:
                    print(f"   ⚠️  Found processed JSON file (will be deleted)")
                
                if not dry_run:
                    # Delete the markdown file
                    os.remove(markdown_path)
                    deleted_markdown_count += 1
                    
                    # Delete the JSON file if it exists
                    if has_json:
                        json_path = processed_paths[paper_hash]
                        os.remove(json_path)
                        deleted_json_count += 1
                        print(f"   ✓ Deleted JSON: {os.path.basename(json_path)}")
                    
                    # Add to skipped list
                    add_to_skipped(paper_hash, f"too_long_{estimated_tokens}_tokens")
                    print(f"   ✓ Deleted markdown and added to skipped list\n")
                else:
                    print(f"   [DRY RUN] Would delete markdown", end="")
                    if has_json:
                        print(f" and JSON file")
                    else:
                        print()
                    print()
        
        except Exception as e:
            print(f"❌ Error processing {paper_hash}: {e}\n")
    
    # Summary statistics
    print(f"\n{'='*60}")
    print(f"Summary")
    print(f"{'='*60}")
    print(f"Total papers analyzed: {total_papers}")
    print(f"Already skipped: {already_skipped_count}")
    print(f"Papers too long: {too_long_count}")
    
    if not dry_run:
        print(f"Markdown files deleted: {deleted_markdown_count}")
        print(f"JSON files deleted: {deleted_json_count}")
    else:
        print(f"Markdown files that would be deleted: {too_long_count}")
        print(f"JSON files that would be deleted: {sum(1 for h in [md_file.replace('.md', '') for md_file in markdown_files] if h in processed_hashes)}")
    
    if size_stats:
        avg_tokens = sum(size_stats) / len(size_stats)
        max_tokens_found = max(size_stats)
        min_tokens_found = min(size_stats)
        
        print(f"\nSize Statistics:")
        print(f"  Average tokens per paper: {avg_tokens:,.0f}")
        print(f"  Min tokens: {min_tokens_found:,}")
        print(f"  Max tokens: {max_tokens_found:,}")
        print(f"  Papers within limit: {sum(1 for t in size_stats if t <= MAX_TOKENS)}")
        print(f"  Papers exceeding limit: {sum(1 for t in size_stats if t > MAX_TOKENS)}")
    
    if dry_run:
        print(f"\n💡 This was a DRY RUN - no files were deleted")
        print(f"   Set dry_run=False to actually delete the files")
    else:
        print(f"\n✓ Files deleted:")
        print(f"   - Markdown files removed from: {MARKDOWN_DIR}")
        print(f"   - JSON files removed from: {OUTPUT_BASE_DIR}/{model_name}/")
        print(f"   - Hashes added to: {SKIPPED_FILE}")
    
    print(f"{'='*60}\n")
    
    return {
        "total": total_papers,
        "too_long": too_long_count,
        "deleted_markdown": deleted_markdown_count if not dry_run else 0,
        "deleted_json": deleted_json_count if not dry_run else 0,
        "already_skipped": already_skipped_count,
        "avg_tokens": sum(size_stats) / len(size_stats) if size_stats else 0,
        "max_tokens": max(size_stats) if size_stats else 0
    }

# For notebook use:
# Run with dry_run=True first to see what would be deleted
stats = filter_long_papers(dry_run=False, model_name="gpt-oss:20b")

# If you're happy with the results, run again with dry_run=False
# stats = filter_long_papers(dry_run=False, model_name="llama3.2")


Filtering Long Papers
Max tokens allowed: 128,000
Max characters: 512,000
Model directory: gpt-oss:20b
Dry run: False

Analyzing 4015 papers...
Found 3205 processed papers in gpt-oss:20b/ directory

⚠️  TOO LONG: 8488454077cb0fd9d31772274c78115d
   Estimated tokens: 179,275 (exceeds 128,000)
   Paper content: 711,695 chars
   Full prompt: 717,101 chars
   ⚠️  Found processed JSON file (will be deleted)
   ✓ Deleted JSON: 8488454077cb0fd9d31772274c78115d.json
   ✓ Deleted markdown and added to skipped list

⚠️  TOO LONG: f49287371916715b9209fa41a275851e
   Estimated tokens: 163,504 (exceeds 128,000)
   Paper content: 648,610 chars
   Full prompt: 654,016 chars
   ⚠️  Found processed JSON file (will be deleted)
   ✓ Deleted JSON: f49287371916715b9209fa41a275851e.json
   ✓ Deleted markdown and added to skipped list

⚠️  TOO LONG: 3cd50f2922b7adaaa9e5113e35bae095
   Estimated tokens: 134,619 (exceeds 128,000)
   Paper content: 533,073 chars
   Full prompt: 538,479 chars
   ⚠️  Found proce

In [17]:
"""
Script to find and remove papers where the author's checklist has "Not Found" answers.
These papers likely have incomplete or malformed checklists.
"""

import os
import re
import json
from pathlib import Path
from datetime import datetime

# Configuration
MARKDOWN_DIR = "neurips_2024_papers/markdown"
OUTPUT_BASE_DIR = "neurips_2024_checklist_analysis"
SKIPPED_FILE = os.path.join(OUTPUT_BASE_DIR, "skipped_papers.txt")
INCOMPLETE_CHECKLIST_FILE = os.path.join(OUTPUT_BASE_DIR, "incomplete_checklist_papers.txt")

def get_all_model_directories():
    """Get list of all model directories in the output base directory."""
    if not os.path.exists(OUTPUT_BASE_DIR):
        return []
    
    model_dirs = []
    for item in os.listdir(OUTPUT_BASE_DIR):
        item_path = os.path.join(OUTPUT_BASE_DIR, item)
        if os.path.isdir(item_path) and not item.startswith('.'):
            model_dirs.append(item)
    
    return model_dirs

def get_processed_papers_all_models():
    """
    Get dictionary of all processed papers across all model directories.
    Returns: {paper_hash: [list of json file paths]}
    """
    all_processed = {}
    model_dirs = get_all_model_directories()
    
    for model_name in model_dirs:
        model_dir = os.path.join(OUTPUT_BASE_DIR, model_name)
        
        for filename in os.listdir(model_dir):
            if filename.endswith('.json'):
                paper_hash = filename.replace('.json', '')
                json_path = os.path.join(model_dir, filename)
                
                if paper_hash not in all_processed:
                    all_processed[paper_hash] = []
                all_processed[paper_hash].append(json_path)
    
    return all_processed

def check_json_for_not_found(json_path):
    """
    Check if a JSON file has any "Not Found" author answers.
    Returns: (has_not_found, not_found_count, not_found_questions)
    """
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        not_found_questions = []
        for question, answers in data.get("questions", {}).items():
            author_answer = answers.get("author_answer", "")
            if author_answer == "Not Found":
                not_found_questions.append(question)
        
        return len(not_found_questions) > 0, len(not_found_questions), not_found_questions
    
    except Exception as e:
        print(f"  ⚠️  Error reading JSON {json_path}: {e}")
        return False, 0, []

def check_markdown_for_not_found(markdown_path):
    """
    Parse markdown and check if checklist has missing answers.
    Returns: (has_not_found, not_found_count, not_found_questions)
    """
    try:
        with open(markdown_path, 'r', encoding='utf-8') as f:
            markdown_text = f.read()
        
        _, author_answers = extract_paper_content_and_checklist(markdown_text)
        
        not_found_questions = []
        for question in CHECKLIST_QUESTIONS:
            if question not in author_answers:
                not_found_questions.append(question)
        
        return len(not_found_questions) > 0, len(not_found_questions), not_found_questions
    
    except Exception as e:
        print(f"  ⚠️  Error reading markdown {markdown_path}: {e}")
        return False, 0, []

def add_to_skipped(paper_hash, reason):
    """Add a paper to the skipped list with reason."""
    os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
    with open(SKIPPED_FILE, 'a') as f:
        f.write(f"{paper_hash}\t{reason}\t{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

def save_incomplete_checklist_hash(paper_hash, not_found_count, questions):
    """Save hash of paper with incomplete checklist."""
    os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
    with open(INCOMPLETE_CHECKLIST_FILE, 'a') as f:
        f.write(f"{paper_hash}\t{not_found_count}\t{','.join(questions)}\t{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

def filter_incomplete_checklists(dry_run=True, check_source="markdown"):
    """
    Find and remove papers with incomplete checklists (Not Found answers).
    
    Args:
        dry_run: If True, only report what would be deleted
        check_source: "markdown" to parse markdown files, "json" to read existing JSONs
    """
    print(f"\n{'='*60}")
    print(f"Filtering Papers with Incomplete Checklists")
    print(f"{'='*60}")
    print(f"Check source: {check_source}")
    print(f"Dry run: {dry_run}")
    print(f"{'='*60}\n")
    
    if not os.path.exists(MARKDOWN_DIR):
        print(f"Error: Markdown directory not found: {MARKDOWN_DIR}")
        return
    
    # Get all markdown files
    markdown_files = [f for f in os.listdir(MARKDOWN_DIR) if f.endswith('.md')]
    
    # Get all processed papers across all models
    all_processed = get_processed_papers_all_models()
    
    # Stats
    total_papers = len(markdown_files)
    incomplete_count = 0
    deleted_markdown_count = 0
    deleted_json_count = 0
    papers_with_issues = []
    
    print(f"Analyzing {total_papers} papers...")
    print(f"Found {len(all_processed)} processed papers across all models")
    print(f"Model directories: {', '.join(get_all_model_directories())}\n")
    
    for md_file in markdown_files:
        paper_hash = md_file.replace('.md', '')
        markdown_path = os.path.join(MARKDOWN_DIR, md_file)
        
        has_not_found = False
        not_found_count = 0
        not_found_questions = []
        
        # Check based on source
        if check_source == "markdown":
            # Parse markdown directly
            has_not_found, not_found_count, not_found_questions = check_markdown_for_not_found(markdown_path)
        
        elif check_source == "json":
            # Check JSON files if they exist
            if paper_hash in all_processed:
                # Check first JSON found (they should all have same author answers)
                first_json = all_processed[paper_hash][0]
                has_not_found, not_found_count, not_found_questions = check_json_for_not_found(first_json)
        
        # If incomplete checklist found
        if has_not_found:
            incomplete_count += 1
            papers_with_issues.append((paper_hash, not_found_count, not_found_questions))
            
            print(f"⚠️  INCOMPLETE: {paper_hash}")
            print(f"   Missing {not_found_count}/{len(CHECKLIST_QUESTIONS)} answers")
            print(f"   Questions not found: {', '.join(not_found_questions[:3])}", end="")
            if len(not_found_questions) > 3:
                print(f" ... (+{len(not_found_questions)-3} more)")
            else:
                print()
            
            # Check if has processed JSONs
            json_files = all_processed.get(paper_hash, [])
            if json_files:
                print(f"   Found {len(json_files)} JSON file(s) across models")
            if not dry_run:
                # Delete markdown
                os.remove(markdown_path)
                deleted_markdown_count += 1
                
                # Delete all JSON files for this paper
                for json_path in json_files:
                    os.remove(json_path)
                    deleted_json_count += 1
                    print(f"   ✓ Deleted JSON: {json_path}")
                
                # Save to incomplete checklist file
                save_incomplete_checklist_hash(paper_hash, not_found_count, not_found_questions)
                
                # Add to skipped list
                add_to_skipped(paper_hash, f"incomplete_checklist_{not_found_count}_missing")
                
                print(f"   ✓ Deleted markdown and all JSONs\n")
            else:
                print(f"   [DRY RUN] Would delete markdown and {len(json_files)} JSON file(s)\n")
    
    # Summary
    print(f"\n{'='*60}")
    print(f"Summary")
    print(f"{'='*60}")
    print(f"Total papers analyzed: {total_papers}")
    print(f"Papers with incomplete checklists: {incomplete_count}")
    
    if not dry_run:
        print(f"Markdown files deleted: {deleted_markdown_count}")
        print(f"JSON files deleted: {deleted_json_count}")
    else:
        total_jsons_to_delete = sum(len(all_processed.get(h, [])) for h, _, _ in papers_with_issues)
        print(f"Markdown files that would be deleted: {incomplete_count}")
        print(f"JSON files that would be deleted: {total_jsons_to_delete}")
    
    if incomplete_count > 0:
        print(f"\nMost common missing question counts:")
        missing_counts = {}
        for _, count, _ in papers_with_issues:
            missing_counts[count] = missing_counts.get(count, 0) + 1
        
        for count in sorted(missing_counts.keys()):
            print(f"  {count} missing: {missing_counts[count]} papers")
    
    if dry_run:
        print(f"\n💡 This was a DRY RUN - no files were deleted")
        print(f"   Set dry_run=False to actually delete the files")
    else:
        print(f"\n✓ Files deleted:")
        print(f"   - Markdown files removed from: {MARKDOWN_DIR}")
        print(f"   - JSON files removed from all model directories")
        print(f"   - Hashes saved to: {INCOMPLETE_CHECKLIST_FILE}")
        print(f"   - Added to skipped list: {SKIPPED_FILE}")
    
    print(f"{'='*60}\n")
    
    return {
        "total": total_papers,
        "incomplete": incomplete_count,
        "deleted_markdown": deleted_markdown_count if not dry_run else 0,
        "deleted_json": deleted_json_count if not dry_run else 0,
        "papers_with_issues": papers_with_issues
    }

stats = filter_incomplete_checklists(dry_run=False, check_source="json")



Filtering Papers with Incomplete Checklists
Check source: json
Dry run: False

Analyzing 4009 papers...
Found 4002 processed papers across all models
Model directories: gpt-oss:20b, deepseek-r1:8b, mistral-nemo:12b, hermes3:8b

⚠️  INCOMPLETE: c40daf14d7a6469e65116507c21faeb7
   Missing 15/15 answers
   Questions not found: Claims, Limitations, Theory Assumptions and Proofs ... (+12 more)
   Found 4 JSON file(s) across models
   ✓ Deleted JSON: neurips_2024_checklist_analysis/gpt-oss:20b/c40daf14d7a6469e65116507c21faeb7.json
   ✓ Deleted JSON: neurips_2024_checklist_analysis/deepseek-r1:8b/c40daf14d7a6469e65116507c21faeb7.json
   ✓ Deleted JSON: neurips_2024_checklist_analysis/mistral-nemo:12b/c40daf14d7a6469e65116507c21faeb7.json
   ✓ Deleted JSON: neurips_2024_checklist_analysis/hermes3:8b/c40daf14d7a6469e65116507c21faeb7.json
   ✓ Deleted markdown and all JSONs

⚠️  INCOMPLETE: f0ebc318e2df08360b2df559e81602e5
   Missing 15/15 answers
   Questions not found: Claims, Limitations, Th